Tasks - Linear Algebra 2 (Medium)

These tasks require combining multiple concepts from the LA-2 notebooks.
Topics: LU decomposition, null space, Gram-Schmidt, projection onto subspace,
Jacobian, MSE gradient, spectral decomposition, matrix square root,
Xavier initialisation, and layer normalisation.

Each task builds on ideas from more than one section.

Task 1 - LU Decomposition and Solve

Decompose a matrix using LU, then use forward and back substitution to solve Ax = b.
Verify the reconstruction P @ A = L @ U and the solution A @ x = b.

In [ ]:
import numpy as np
from scipy.linalg import lu, solve_triangular

A = np.array([[3.0, 1.0, 2.0],
              [6.0, 3.0, 8.0],
              [3.0, 4.0, 5.0]])
b = np.array([1.0, 2.0, 3.0])

# LU decomposition
P, L, U = None  # scipy.linalg.lu(A)

# verify P @ A = L @ U
print(f'P @ A = L @ U: {np.allclose(P @ A, L @ U)}')

# solve Ax = b: first solve Ly = Pb, then solve Ux = y
Pb = None  # P @ b
y  = None  # solve_triangular(L, Pb, lower=True)
x  = None  # solve_triangular(U, y,  lower=False)

print(f'solution x = {x.round(4)}')
print(f'A @ x = b  : {np.allclose(A @ x, b)}')

Task 2 - Find the Null Space

Find the null space of a rank-deficient matrix via SVD.
The null space consists of the rows of V.T corresponding to near-zero singular values.
Verify: A @ null_vector should be approximately zero.

In [ ]:
import numpy as np

# col 3 = 2*col 1 + col 2 -- rank 2, nullity 1
A = np.array([[1.0, 2.0,  4.0],
              [3.0, 1.0,  7.0],
              [2.0, 4.0,  8.0]])

rank = None  # np.linalg.matrix_rank(A)
print(f'rank = {rank}  (expected 2)')

# SVD to find null space
U, S, Vt = None  # np.linalg.svd(A)
print(f'singular values: {S.round(6)}')

# the last row of Vt (smallest singular value) spans the null space
null_vec = None  # Vt[-1]

print(f'null vector: {null_vec.round(4)}')
print(f'A @ null_vec: {(A @ null_vec).round(8)}  (should be ~0)')

Task 3 - Gram-Schmidt Orthogonalisation

Apply Gram-Schmidt to three linearly independent vectors to produce an orthonormal basis.
At each step, subtract all projections onto the previously found basis vectors, then normalise.
Verify: all dot products between basis vectors are zero, all norms are 1.

In [ ]:
import numpy as np

v1 = np.array([3.0, 1.0, 0.0])
v2 = np.array([2.0, 2.0, 1.0])
v3 = np.array([1.0, 0.0, 3.0])

# step 1: normalise v1
q1 = None  # v1 / np.linalg.norm(v1)

# step 2: remove q1 component from v2, then normalise
v2_perp = None  # v2 - np.dot(v2, q1) * q1
q2 = None       # v2_perp / np.linalg.norm(v2_perp)

# step 3: remove q1 and q2 components from v3, then normalise
v3_perp = None  # v3 - np.dot(v3, q1)*q1 - np.dot(v3, q2)*q2
q3 = None       # v3_perp / np.linalg.norm(v3_perp)

Q = np.column_stack([q1, q2, q3])
print(f'Q.T @ Q (should be identity):\n{(Q.T @ Q).round(6)}')
print(f'all norms = 1: {np.allclose([np.linalg.norm(q1), np.linalg.norm(q2), np.linalg.norm(q3)], 1)}')

Task 4 - Orthogonal Projection onto a Subspace

Project vector b onto the column space of matrix A.
For a matrix A with orthonormal columns Q, the projection is P = Q @ Q.T
For a general A, the projection matrix is P = A @ (A.T @ A)^-1 @ A.T
Verify the residual is perpendicular to the column space.

In [ ]:
import numpy as np

# column space spanned by two vectors in R^3
A = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])  # shape (3, 2)

b = np.array([2.0, 3.0, 1.0])

# projection matrix: P = A (A.T A)^-1 A.T
P = None  # A @ np.linalg.inv(A.T @ A) @ A.T

# projected b
b_proj = None  # P @ b

# residual
residual = None  # b - b_proj

print(f'b        = {b}')
print(f'b_proj   = {b_proj.round(4)}')
print(f'residual = {residual.round(4)}')
print(f'residual perp to col space: {np.allclose(A.T @ residual, 0, atol=1e-10)}')
print(f'b_proj + residual = b: {np.allclose(b_proj + residual, b)}')

Task 5 - Gradient of MSE Loss

Compute the gradient of the MSE loss L = (1/n)||Xw - y||^2 with respect to w.
Analytical formula: dL/dw = (2/n) X^T (Xw - y)
Verify against the numerical gradient via finite differences.

In [ ]:
import numpy as np

np.random.seed(0)
n, d = 20, 3
X = np.random.randn(n, d)
y = np.random.randn(n)
w = np.array([0.5, -0.3, 0.8])

# analytical gradient: (2/n) * X.T @ (Xw - y)
residual = None  # X @ w - y
grad_anal = None  # (2/n) * X.T @ residual

# numerical gradient via central finite differences
eps = 1e-5
def mse(w_):
    return ((X @ w_ - y)**2).mean()

grad_num = np.array([(mse(w + eps*np.eye(d)[i]) - mse(w - eps*np.eye(d)[i])) / (2*eps)
                     for i in range(d)])

print(f'analytical gradient: {grad_anal.round(6)}')
print(f'numerical  gradient: {grad_num.round(6)}')
print(f'match: {np.allclose(grad_anal, grad_num, atol=1e-5)}')

Task 6 - Jacobian via PyTorch Autograd

Compute the Jacobian of a vector-valued function f: R^3 -> R^2 using PyTorch autograd.
Verify against the analytical Jacobian.

In [ ]:
import torch

# f(x) = [x0*x1 + x2^2,  x0^2 - x1*x2]
# J = [[x1, x0, 2*x2], [2*x0, -x2, -x1]]  at x = [1, 2, 3]

def f(x):
    return torch.stack([x[0]*x[1] + x[2]**2,
                        x[0]**2 - x[1]*x[2]])

x = torch.tensor([1.0, 2.0, 3.0])

# compute Jacobian using torch.autograd.functional.jacobian
J = None  # torch.autograd.functional.jacobian(f, x)

# analytical at x = [1, 2, 3]
J_anal = torch.tensor([[2.0, 1.0, 6.0],
                        [2.0, -3.0, -2.0]])

print(f'computed Jacobian:\n{J}')
print(f'analytical:\n{J_anal}')
print(f'match: {torch.allclose(J, J_anal)}')

Task 7 - Spectral Decomposition

Compute the spectral decomposition of a symmetric matrix: A = Q diag(lambda) Q^T
Then verify the reconstruction and use it to compute A raised to the power 3.

In [ ]:
import numpy as np

A = np.array([[4.0, 2.0, 1.0],
              [2.0, 3.0, 1.0],
              [1.0, 1.0, 2.0]])

# spectral decomposition (use eigh for symmetric matrices)
eigenvalues, Q = None  # np.linalg.eigh(A)

# reconstruct A = Q diag(lambda) Q.T
A_recon = None  # Q @ np.diag(eigenvalues) @ Q.T
print(f'reconstruction error: {np.linalg.norm(A - A_recon):.2e}')

# compute A^3 using spectral decomposition: Q diag(lambda^3) Q.T
A_cubed_spectral = None  # Q @ np.diag(eigenvalues**3) @ Q.T

# verify against direct computation
A_cubed_direct = A @ A @ A
print(f'A^3 spectral vs direct match: {np.allclose(A_cubed_spectral, A_cubed_direct, atol=1e-8)}')
print(f'A^3:\n{A_cubed_spectral.round(4)}')

Task 8 - Matrix Square Root

Compute the square root of a positive definite matrix using spectral decomposition.
A^(1/2) = Q diag(sqrt(lambda)) Q.T
Verify: A^(1/2) @ A^(1/2) = A

In [ ]:
import numpy as np

A = np.array([[5.0, 4.0],
              [4.0, 5.0]])

# spectral decomposition
eigenvalues, Q = None  # np.linalg.eigh(A)

# matrix square root: Q diag(sqrt(lambda)) Q.T
A_sqrt = None  # your code here

# verify A^(1/2) @ A^(1/2) = A
check = None  # A_sqrt @ A_sqrt

print(f'A:\n{A}')
print(f'A_sqrt:\n{A_sqrt.round(4)}')
print(f'A_sqrt @ A_sqrt:\n{check.round(4)}')
print(f'equals A: {np.allclose(check, A, atol=1e-8)}')

# verify using scipy
from scipy.linalg import sqrtm
print(f'matches scipy sqrtm: {np.allclose(A_sqrt, sqrtm(A).real, atol=1e-8)}')

Task 9 - Xavier Initialisation Variance Check

Xavier initialisation uses std = sqrt(2 / (fan_in + fan_out)).
Generate a weight matrix with Xavier init and verify its variance.
Then pass random input through one tanh layer and check that output variance
is close to the input variance.

In [ ]:
import numpy as np

np.random.seed(42)
fan_in, fan_out = 128, 64

# Xavier standard deviation
std_xavier = None  # sqrt(2 / (fan_in + fan_out))

# initialise weight matrix
W = None  # np.random.randn(fan_out, fan_in) * std_xavier

print(f'Xavier std      : {std_xavier:.6f}')
print(f'W actual std    : {W.std():.6f}')
print(f'W actual var    : {W.var():.6f}')
print(f'expected var    : {2/(fan_in + fan_out):.6f}')

# pass 1000 samples through one layer
X_input = np.random.randn(1000, fan_in)
X_output = np.tanh(X_input @ W.T)   # (1000, fan_out)

print(f'\ninput  variance : {X_input.var():.4f}')
print(f'output variance : {X_output.var():.4f}  (should be close to input var)')

Task 10 - Layer Normalisation from Scratch

Implement layer normalisation for a batch of token embeddings (batch, seq, d_model).
Normalise each token (position) across its d_model features independently.
After normalisation, each token should have mean 0 and std 1.

In [ ]:
import numpy as np

np.random.seed(7)
batch, seq, d_model = 4, 6, 32
X = np.random.randn(batch, seq, d_model) * 5 + 3   # non-zero mean and varied scale

gamma = np.ones(d_model)   # learnable scale (start at 1)
beta  = np.zeros(d_model)  # learnable shift (start at 0)
eps   = 1e-5

# compute mean over last axis (features) for each token
mu = None  # X.mean(axis=-1, keepdims=True)

# compute std over last axis
sigma = None  # X.std(axis=-1, keepdims=True)

# normalise
X_norm = None  # (X - mu) / (sigma + eps)

# apply learnable scale and shift
X_out = None  # gamma * X_norm + beta

print(f'input mean per token (first batch): {X[0].mean(axis=-1).round(2)}')
print(f'output mean per token: {X_out[0].mean(axis=-1).round(6)}  (should be ~0)')
print(f'output std  per token: {X_out[0].std(axis=-1).round(4)}   (should be ~1)')